# G3C Qwen retrieval - DEV R0-R4

This notebook runs only the manifest-selected G3C GPU retrieval workload. Enable a Kaggle GPU and Internet before Run all. The payload validator rejects gold/evaluator files; the promotion payload additionally binds exactly one dev-selected stage.

Models are loaded sequentially in FP16 with SDPA. Do not edit model revisions, instructions, thresholds, or stage settings in this notebook.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
EXPECTED_MODE = "dev"
matches = sorted(Path('/kaggle/input').rglob('g3c_gpu_payload_manifest.json'))
assert len(matches) == 1, f'expected exactly one G3C payload, found {matches}'
PAYLOAD = matches[0].parent
manifest = json.loads(matches[0].read_text(encoding='utf-8'))
assert manifest['mode'] == EXPECTED_MODE, (manifest['mode'], EXPECTED_MODE)
requirements = PAYLOAD / manifest['paths']['requirements']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print({'payload': str(PAYLOAD), 'mode': manifest['mode'], 'questions': manifest['question_count'], 'selected_stage': manifest.get('selected_stage'), 'fingerprint': manifest['payload_fingerprint']})


In [ ]:
os.environ['HF_HOME'] = '/kaggle/temp/g3c_hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
sys.dont_write_bytecode = True
sys.path.insert(0, str(PAYLOAD / 'code'))
import torch, transformers
from vifinqa.g3c.payload import validate_gpu_payload
assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'
assert transformers.__version__ == '4.53.3', transformers.__version__
validated = validate_gpu_payload(PAYLOAD)
assert validated['protocol_fingerprint'] == manifest['protocol_fingerprint']
torch.cuda.reset_peak_memory_stats()
print({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'transformers': transformers.__version__, 'payload_valid': validated['payload_fingerprint']})


In [ ]:
OUT = Path('/kaggle/working/g3c_dev_run')
runner = PAYLOAD / manifest['paths']['runner']
command = [sys.executable, str(runner), '--payload', str(PAYLOAD), '--out-dir', str(OUT), '--backend', 'qwen']
print('Starting pinned Qwen G3C run:', command)
subprocess.run(command, check=True)


In [ ]:
from vifinqa.g3c.validate import validate_gpu_results
validation_path = Path('/kaggle/working/g3c_dev_run_import_validation.json')
report = validate_gpu_results(payload_dir=PAYLOAD, result_dir=OUT, output_path=validation_path, require_scientific=True)
print(json.dumps(report, ensure_ascii=False, indent=2))


In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/g3c_dev_results', 'zip', root_dir=OUT)
print({'download_directory': str(OUT), 'download_zip': archive, 'validation': str(validation_path)})
